# Notebook 02 — Fingerprint Baseline Models
## ECFP4 + Random Forest / XGBoost on BBBP and ESOL

**Author:** Shahid Afridi Laskar  
**Project:** ChemBERTa-ADMET  

---

### Purpose

Before fine-tuning ChemBERTa, we establish strong classical baselines using:
- **ECFP4** Morgan fingerprints (radius=2, 2048 bits) — the most widely used molecular representation
- **Random Forest** and **XGBoost** classifiers/regressors

These baselines define the performance bar that ChemBERTa must beat to justify the added complexity of a transformer.

**Evaluation:**
- Classification (BBBP): ROC-AUC, PR-AUC
- Regression (ESOL): RMSE, R²

---

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier, XGBRegressor
import deepchem as dc
import warnings
warnings.filterwarnings('ignore')

from src.data_utils import dataset_to_dataframe, smiles_list_to_ecfp4
from src.evaluate import classification_metrics, regression_metrics, plot_roc_curves, plot_regression_scatter

print('Ready.')

## 1. Load and Featurise — BBBP (Classification)

In [ ]:
# Load BBBP
tasks_bbbp, datasets_bbbp, _ = dc.molnet.load_bbbp(
    featurizer=dc.feat.DummyFeaturizer(), splitter='scaffold')
train_bbbp, val_bbbp, test_bbbp = datasets_bbbp

df_train = dataset_to_dataframe(train_bbbp, tasks_bbbp)
df_test  = dataset_to_dataframe(test_bbbp,  tasks_bbbp)

# ECFP4 fingerprints
X_train, valid_idx_train = smiles_list_to_ecfp4(df_train['smiles'].tolist())
X_test,  valid_idx_test  = smiles_list_to_ecfp4(df_test['smiles'].tolist())

y_train = df_train['p_np'].values[valid_idx_train]
y_test  = df_test['p_np'].values[valid_idx_test]

print(f'Train: X={X_train.shape}, y={y_train.shape}')
print(f'Test:  X={X_test.shape},  y={y_test.shape}')
print(f'Class balance (train): {y_train.mean():.2f} positive')

## 2. Train Baseline Classifiers — BBBP

In [ ]:
# Random Forest
rf_clf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
rf_clf.fit(X_train, y_train)
rf_probs = rf_clf.predict_proba(X_test)[:, 1]

# XGBoost
xgb_clf = XGBClassifier(n_estimators=200, learning_rate=0.05,
                         use_label_encoder=False, eval_metric='logloss',
                         random_state=42, verbosity=0)
xgb_clf.fit(X_train, y_train)
xgb_probs = xgb_clf.predict_proba(X_test)[:, 1]

# Metrics
rf_metrics  = classification_metrics(y_test, rf_probs)
xgb_metrics = classification_metrics(y_test, xgb_probs)

print('Random Forest:', rf_metrics)
print('XGBoost:',       xgb_metrics)

In [ ]:
# ROC curves
plot_roc_curves(
    y_true_dict={'RF':  y_test, 'XGBoost': y_test},
    y_pred_dict={'RF':  rf_probs, 'XGBoost': xgb_probs},
    title='BBBP — ECFP4 Baseline ROC Curves',
    save_path='../figures/02_bbbp_roc.png'
)

## 3. Load and Featurise — ESOL (Regression)

In [ ]:
# Load ESOL
tasks_esol, datasets_esol, _ = dc.molnet.load_delaney(
    featurizer=dc.feat.DummyFeaturizer(), splitter='scaffold')
train_esol, val_esol, test_esol = datasets_esol

df_train_e = dataset_to_dataframe(train_esol, tasks_esol)
df_test_e  = dataset_to_dataframe(test_esol,  tasks_esol)

target_col = tasks_esol[0]

X_train_e, vi_train = smiles_list_to_ecfp4(df_train_e['smiles'].tolist())
X_test_e,  vi_test  = smiles_list_to_ecfp4(df_test_e['smiles'].tolist())

y_train_e = df_train_e[target_col].values[vi_train]
y_test_e  = df_test_e[target_col].values[vi_test]

print(f'ESOL Train: {X_train_e.shape}, Test: {X_test_e.shape}')

## 4. Train Baseline Regressors — ESOL

In [ ]:
# Random Forest
rf_reg = RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)
rf_reg.fit(X_train_e, y_train_e)
rf_preds = rf_reg.predict(X_test_e)

# XGBoost
xgb_reg = XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42, verbosity=0)
xgb_reg.fit(X_train_e, y_train_e)
xgb_preds = xgb_reg.predict(X_test_e)

# Metrics
rf_reg_m  = regression_metrics(y_test_e, rf_preds)
xgb_reg_m = regression_metrics(y_test_e, xgb_preds)

print('Random Forest Regression:', rf_reg_m)
print('XGBoost Regression:',       xgb_reg_m)

In [ ]:
plot_regression_scatter(
    y_test_e, rf_preds,
    label='RF ECFP4',
    xlabel='Measured logS',
    ylabel='Predicted logS',
    save_path='../figures/02_esol_rf_scatter.png'
)

## 5. Baseline Summary Table

In [ ]:
summary = pd.DataFrame([
    {'Dataset': 'BBBP',  'Model': 'RF  + ECFP4',  'ROC-AUC': rf_metrics['ROC_AUC'],   'PR-AUC': rf_metrics['PR_AUC'],   'RMSE': '-',                  'R2': '-'},
    {'Dataset': 'BBBP',  'Model': 'XGB + ECFP4',  'ROC-AUC': xgb_metrics['ROC_AUC'],  'PR-AUC': xgb_metrics['PR_AUC'],  'RMSE': '-',                  'R2': '-'},
    {'Dataset': 'ESOL',  'Model': 'RF  + ECFP4',  'ROC-AUC': '-',                       'PR-AUC': '-',                    'RMSE': rf_reg_m['RMSE'],     'R2': rf_reg_m['R2']},
    {'Dataset': 'ESOL',  'Model': 'XGB + ECFP4',  'ROC-AUC': '-',                       'PR-AUC': '-',                    'RMSE': xgb_reg_m['RMSE'],    'R2': xgb_reg_m['R2']},
    {'Dataset': 'BBBP',  'Model': 'ChemBERTa',    'ROC-AUC': '(notebook 03)',            'PR-AUC': '(notebook 03)',        'RMSE': '-',                  'R2': '-'},
    {'Dataset': 'ESOL',  'Model': 'ChemBERTa',    'ROC-AUC': '-',                       'PR-AUC': '-',                    'RMSE': '(notebook 03)',       'R2': '(notebook 03)'},
])

print(summary.to_string(index=False))

## Summary

ECFP4 + RF/XGBoost gives a solid baseline. These numbers are the performance bar ChemBERTa needs to beat in **Notebook 03**.

Expected literature benchmark values (MoleculeNet leaderboard):
- BBBP ROC-AUC: ~0.88–0.92 for best models
- ESOL RMSE: ~0.5–0.6 for best models

**Next:** `03_chemberta_finetuning.ipynb` — Fine-tune ChemBERTa on BBBP and ESOL.